# 출시 전 LLM 체크리스트용 데이터 생성

이 노트북은 보고서용 그래프를 만드는 노트북이 아니다.  
`03_run_llm_allgames_analysi.ipynb`에서 생성한 LLM 분석 결과를 읽고,  
나중에 LLM이 출시 전 체크리스트를 만들 때 사용할 CSV 파일을 생성한다.

사용자 조건 입력과 LLM 호출은 이 노트북에서 하지 않는다.

# 1. 목적

최종 목표는 개발자가 입력한 장르, 가격대, Steam 태그, 플레이 방식에 따라  
출시 전 점검 항목을 LLM이 생성할 수 있도록 근거 데이터를 미리 정리하는 것이다.

이 노트북에서는 다음 4개 CSV만 생성한다.

| 파일명 | 역할 |
|---|---|
| `prelaunch_game_base.csv` | 게임 단위 기본 정보 |
| `prelaunch_issue_repeat_summary.csv` | 전체 이슈 반복성 요약 |
| `prelaunch_condition_issue_summary.csv` | 조건별 이슈 요약 |
| `prelaunch_checklist_evidence_base.csv` | LLM 입력용 근거 문장 데이터 |

# 2. 기본 설정

In [52]:
# ============================================================
# 기본 라이브러리
# ============================================================
import ast
import platform
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
from IPython.display import display



# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

In [53]:
# ============================================================
# 경로 설정
# ============================================================
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정한다.
ROOT = Path.cwd()

# Jupyter 실행 위치가 하위 폴더일 수 있으므로,
# data/preprocessed 폴더가 보일 때까지 상위 폴더를 탐색한다.
if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# 03_run_llm_allgames_analysi.ipynb에서 사용한 실행 이름과 맞춘다.
RUN_NAME = "main_allgames_d0-d30"
OUTPUT_DIR = ROOT / "data" / "outputs" / RUN_NAME

# LLM 실행 산출물 (입력 파일)
RESULT_CSV_PATH = OUTPUT_DIR / "llm_review_analysis_result.csv"
ISSUE_TAG_FLAT_PATH = OUTPUT_DIR / "llm_issue_tags_flat.csv"
LLM_INPUT_PATH = OUTPUT_DIR / "llm_input_reviews.csv"
GRADED_GAMES_PATH = ROOT / "data" / "preprocessed" / "steam_indie_games_graded.csv"

# 이번 노트북에서 생성할 CSV 저장 폴더
CHECKLIST_DATA_DIR = OUTPUT_DIR / "prelaunch_checklist_data"
CHECKLIST_DATA_DIR.mkdir(parents=True, exist_ok=True)

# 출력 파일
GAME_BASE_PATH = CHECKLIST_DATA_DIR / "prelaunch_game_base.csv"
ISSUE_REPEAT_SUMMARY_PATH = CHECKLIST_DATA_DIR / "prelaunch_issue_repeat_summary.csv"
CONDITION_ISSUE_SUMMARY_PATH = CHECKLIST_DATA_DIR / "prelaunch_condition_issue_summary.csv"
CHECKLIST_EVIDENCE_BASE_PATH = CHECKLIST_DATA_DIR / "prelaunch_checklist_evidence_base.csv"

In [54]:
# ============================================================
# 분석 기준 설정
# ============================================================
# 튜터님 피드백 반영:
# 리뷰에서 몇 번 나왔는지가 아니라,
# 몇 개 게임에서 반복적으로 나타났는지를 중심으로 본다.

GAME_ID_COL = "appid"
GAME_NAME_COL = "game_name"
REVIEW_ID_COL = "recommendationid"
ISSUE_COL = "issue_name_kor"

# 우선순위 기준
HIGH_PRIORITY_MIN_NEGATIVE_GAMES = 5
HIGH_PRIORITY_MIN_URGENCY_GAMES = 3
HIGH_PRIORITY_MIN_RATIO = 30.0

MID_PRIORITY_MIN_ISSUE_GAMES = 3
MID_PRIORITY_MIN_NEGATIVE_GAMES = 2

# LLM 입력용 근거표에서 조건별로 남길 최대 이슈 수
TOP_N_EVIDENCE_PER_CONDITION = 15

# 3. 데이터 불러오기 및 기본 전처리

In [55]:
# ============================================================
# 데이터 불러오기
# ============================================================
review_df = pd.read_csv(RESULT_CSV_PATH)
tag_df = pd.read_csv(ISSUE_TAG_FLAT_PATH)
llm_input_df = pd.read_csv(LLM_INPUT_PATH)

In [56]:
# ============================================================
# 기본 컬럼 타입 정리
# ============================================================
review_df[REVIEW_ID_COL] = review_df[REVIEW_ID_COL].astype(str)
tag_df[REVIEW_ID_COL] = tag_df[REVIEW_ID_COL].astype(str)
llm_input_df[REVIEW_ID_COL] = llm_input_df[REVIEW_ID_COL].astype(str)

review_df[GAME_ID_COL] = review_df[GAME_ID_COL].astype(int)
tag_df[GAME_ID_COL] = tag_df[GAME_ID_COL].astype(int)
llm_input_df[GAME_ID_COL] = llm_input_df[GAME_ID_COL].astype(int)

review_df["llm_sentiment"] = review_df["llm_sentiment"].astype(str).str.strip().str.lower()
review_df["steam_label_text"] = review_df["steam_label_text"].astype(str).str.strip().str.lower()
review_df["urgency"] = review_df["urgency"].astype(str).str.strip().str.lower()

tag_df["llm_sentiment"] = tag_df["llm_sentiment"].astype(str).str.strip().str.lower()
tag_df["steam_label_text"] = tag_df["steam_label_text"].astype(str).str.strip().str.lower()
tag_df["tag_sentiment"] = tag_df["tag_sentiment"].astype(str).str.strip().str.lower()
tag_df["urgency"] = tag_df["urgency"].astype(str).str.strip().str.lower()
tag_df[ISSUE_COL] = tag_df[ISSUE_COL].astype(str).str.strip()

In [57]:
# ============================================================
# LLM 입력 파일의 메타데이터 결합
# ============================================================
# LLM 결과 파일에 가격대, 장르, Steam 태그, 카테고리 정보가 빠질 수 있으므로
# llm_input_reviews.csv에서 필요한 메타데이터를 다시 붙인다.

meta_cols = [
    REVIEW_ID_COL,
    "price",
    "price_group",
    "genres_text",
    "categories_text",
    "top_steam_tags_text"
]

meta_df = llm_input_df[meta_cols].drop_duplicates(REVIEW_ID_COL)

add_cols = ["price", "price_group", "genres_text", "categories_text", "top_steam_tags_text"]

review_df = review_df.drop(columns=add_cols, errors="ignore")
review_df = review_df.merge(meta_df, on=REVIEW_ID_COL, how="left")

tag_df = tag_df.drop(columns=add_cols, errors="ignore")
tag_df = tag_df.merge(
    review_df[[REVIEW_ID_COL] + add_cols].drop_duplicates(REVIEW_ID_COL),
    on=REVIEW_ID_COL,
    how="left"
)

In [58]:
# ============================================================
# 플레이 방식 간단 분류
# ============================================================
# categories_text에 Co-op, Multi-player, Online이 들어가면
# 멀티/협동 요소가 있는 게임으로 분류한다.
# 그 외는 Single-player 중심으로 분류한다.

review_df["categories_text"] = review_df["categories_text"].fillna("").astype(str)

review_df["play_style"] = "Single-player 중심"

review_df.loc[
    review_df["categories_text"].str.contains("Co-op|Multi-player|Online", case=False, regex=True),
    "play_style"
] = "멀티/협동 요소 포함"

tag_df = tag_df.merge(
    review_df[[REVIEW_ID_COL, "play_style"]].drop_duplicates(REVIEW_ID_COL),
    on=REVIEW_ID_COL,
    how="left"
)

In [59]:
# ============================================================
# 분석에 필요한 플래그 생성
# ============================================================
review_df["is_llm_positive"] = review_df["llm_sentiment"].eq("positive")
review_df["is_llm_negative"] = review_df["llm_sentiment"].eq("negative")
review_df["is_llm_mixed"] = review_df["llm_sentiment"].eq("mixed")
review_df["is_high_urgency"] = review_df["urgency"].eq("high")

tag_df["is_tag_positive"] = tag_df["tag_sentiment"].eq("positive")
tag_df["is_tag_negative"] = tag_df["tag_sentiment"].eq("negative")
tag_df["is_tag_mixed"] = tag_df["tag_sentiment"].eq("mixed")
tag_df["is_high_urgency"] = tag_df["urgency"].eq("high")

# 4. 게임 단위 기본 정보 생성

In [60]:
# ============================================================
# prelaunch_game_base.csv 생성
# ============================================================
# 1행 = 게임 1개
# 나중에 LLM 코드에서 사용자 조건에 해당하는 게임 수를 확인할 때 사용한다.

game_meta_cols = [
    GAME_ID_COL,
    GAME_NAME_COL,
    "genres_text",
    "price_group",
    "top_steam_tags_text",
    "categories_text",
    "play_style"
]

game_base = (
    review_df
    .groupby(game_meta_cols, dropna=False)
    .agg(
        review_count=(REVIEW_ID_COL, "nunique"),
        llm_positive_count=("is_llm_positive", "sum"),
        llm_negative_count=("is_llm_negative", "sum"),
        llm_mixed_count=("is_llm_mixed", "sum"),
        high_urgency_count=("is_high_urgency", "sum")
    )
    .reset_index()
)

game_base["llm_positive_ratio"] = (game_base["llm_positive_count"] / game_base["review_count"] * 100).round(1)
game_base["llm_negative_ratio"] = (game_base["llm_negative_count"] / game_base["review_count"] * 100).round(1)
game_base["high_urgency_ratio"] = (game_base["high_urgency_count"] / game_base["review_count"] * 100).round(1)

# 5. 게임-이슈 단위 내부 집계

이 단계에서 만드는 `game_issue_df`는 내부 계산용 데이터다.  
최종 CSV로 저장하지 않는다.

In [61]:
# ============================================================
# 리뷰-이슈 단위 중복 제거
# ============================================================
# 같은 리뷰 안에서 같은 이슈가 중복으로 들어간 경우 1번만 본다.

tag_base = tag_df.dropna(subset=[GAME_ID_COL, REVIEW_ID_COL, ISSUE_COL]).copy()
tag_base = tag_base[tag_base[ISSUE_COL] != ""]
tag_base = tag_base[tag_base[ISSUE_COL].str.lower() != "nan"]

review_issue_df = (
    tag_base
    .drop_duplicates(subset=[REVIEW_ID_COL, ISSUE_COL, "tag_sentiment"])
    .copy()
)

In [62]:
# ============================================================
# 게임-이슈 단위 테이블 생성
# ============================================================
# 같은 게임 안에서 같은 이슈가 여러 리뷰에 반복되어도,
# 최종 중요도 계산에서는 "해당 게임에서 이슈가 발생했다"로 본다.

game_issue_df = (
    review_issue_df
    .groupby([GAME_ID_COL, GAME_NAME_COL, ISSUE_COL])
    .agg(
        issue_review_count=(REVIEW_ID_COL, "nunique"),
        positive_review_count=("is_tag_positive", "sum"),
        negative_review_count=("is_tag_negative", "sum"),
        mixed_review_count=("is_tag_mixed", "sum"),
        high_urgency_review_count=("is_high_urgency", "sum")
    )
    .reset_index()
)

game_issue_df["issue_present"] = game_issue_df["issue_review_count"] > 0
game_issue_df["positive_present"] = game_issue_df["positive_review_count"] > 0
game_issue_df["negative_present"] = game_issue_df["negative_review_count"] > 0
game_issue_df["mixed_present"] = game_issue_df["mixed_review_count"] > 0
game_issue_df["high_urgency_present"] = game_issue_df["high_urgency_review_count"] > 0

game_issue_df["both_positive_negative_present"] = (
    game_issue_df["positive_present"] & game_issue_df["negative_present"]
)

# 6. 전체 이슈 반복성 요약 생성

In [63]:
# ============================================================
# 우선순위 함수
# ============================================================
# 상: 부정 또는 High urgency가 여러 게임에서 반복되는 이슈
# 중: 일정 수 이상의 게임에서 반복되는 이슈
# 하: 참고용 이슈

def get_priority_level(issue_game_count, issue_game_ratio, negative_game_count, high_urgency_game_count):
    if (
        high_urgency_game_count >= HIGH_PRIORITY_MIN_URGENCY_GAMES
        or negative_game_count >= HIGH_PRIORITY_MIN_NEGATIVE_GAMES
        or issue_game_ratio >= HIGH_PRIORITY_MIN_RATIO
    ):
        return "상"
    elif (
        issue_game_count >= MID_PRIORITY_MIN_ISSUE_GAMES
        or negative_game_count >= MID_PRIORITY_MIN_NEGATIVE_GAMES
    ):
        return "중"
    else:
        return "하"


def get_priority_reason(row):
    if row["priority_level"] == "상":
        return "여러 게임에서 반복되었고 부정 또는 High urgency 비중이 높아 출시 전 우선 점검이 필요함"
    elif row["priority_level"] == "중":
        return "일부 게임에서 반복되어 출시 전 확인이 필요함"
    else:
        return "반복성은 낮지만 참고용으로 확인할 수 있음"

In [64]:
# ============================================================
# prelaunch_issue_repeat_summary.csv 생성
# ============================================================
# 전체 게임 기준 이슈 반복성 요약표

total_game_count = game_base[GAME_ID_COL].nunique()

issue_repeat_summary = (
    game_issue_df
    .groupby(ISSUE_COL)
    .agg(
        issue_game_count=(GAME_ID_COL, "nunique"),
        positive_game_count=("positive_present", "sum"),
        negative_game_count=("negative_present", "sum"),
        mixed_game_count=("mixed_present", "sum"),
        high_urgency_game_count=("high_urgency_present", "sum"),
        both_positive_negative_game_count=("both_positive_negative_present", "sum"),
        total_issue_review_count=("issue_review_count", "sum")
    )
    .reset_index()
)

issue_repeat_summary["total_game_count"] = total_game_count

issue_repeat_summary["issue_game_ratio"] = (
    issue_repeat_summary["issue_game_count"] / issue_repeat_summary["total_game_count"] * 100
).round(1)

issue_repeat_summary["priority_level"] = issue_repeat_summary.apply(
    lambda row: get_priority_level(
        row["issue_game_count"],
        row["issue_game_ratio"],
        row["negative_game_count"],
        row["high_urgency_game_count"]
    ),
    axis=1
)

issue_repeat_summary["priority_reason"] = issue_repeat_summary.apply(get_priority_reason, axis=1)

issue_repeat_summary = issue_repeat_summary.sort_values(
    ["priority_level", "issue_game_count", "negative_game_count", "high_urgency_game_count"],
    ascending=[True, False, False, False]
)

# 7. 조건별 이슈 요약 생성

In [65]:
# ============================================================
# 문자열 리스트 분리 함수
# ============================================================
# "Action, Indie, RPG" 같은 문자열을 ["Action", "Indie", "RPG"] 형태로 바꾼다.

def split_text_list(text):
    text = str(text)
    text = text.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    values = [value.strip() for value in text.split(",")]
    values = [value for value in values if value and value.lower() != "nan"]
    return values

In [66]:
# ============================================================
# 조건별 게임 목록 생성
# ============================================================
# 1행 = 게임 1개가 어떤 조건값을 가지고 있는지

game_condition_parts = []

# 가격대
price_part = game_base[[GAME_ID_COL, "price_group"]].drop_duplicates().copy()
price_part = price_part.rename(columns={"price_group": "condition_value"})
price_part["condition_type"] = "price_group"
game_condition_parts.append(price_part[[GAME_ID_COL, "condition_type", "condition_value"]])

# 플레이 방식
play_part = game_base[[GAME_ID_COL, "play_style"]].drop_duplicates().copy()
play_part = play_part.rename(columns={"play_style": "condition_value"})
play_part["condition_type"] = "play_style"
game_condition_parts.append(play_part[[GAME_ID_COL, "condition_type", "condition_value"]])

# 장르
genre_part = game_base[[GAME_ID_COL, "genres_text"]].drop_duplicates().copy()
genre_part["condition_value"] = genre_part["genres_text"].apply(split_text_list)
genre_part = genre_part.explode("condition_value")
genre_part["condition_type"] = "genre"
game_condition_parts.append(genre_part[[GAME_ID_COL, "condition_type", "condition_value"]])

# Steam 태그
tag_part = game_base[[GAME_ID_COL, "top_steam_tags_text"]].drop_duplicates().copy()
tag_part["condition_value"] = tag_part["top_steam_tags_text"].apply(split_text_list)
tag_part = tag_part.explode("condition_value")
tag_part["condition_type"] = "steam_tag"
game_condition_parts.append(tag_part[[GAME_ID_COL, "condition_type", "condition_value"]])

game_condition_df = pd.concat(game_condition_parts, ignore_index=True)
game_condition_df["condition_value"] = game_condition_df["condition_value"].astype(str).str.strip()
game_condition_df = game_condition_df[game_condition_df["condition_value"] != ""]
game_condition_df = game_condition_df[game_condition_df["condition_value"].str.lower() != "nan"]

In [67]:
# ============================================================
# 조건별 전체 게임 수 계산
# ============================================================
condition_game_count = (
    game_condition_df
    .groupby(["condition_type", "condition_value"])[GAME_ID_COL]
    .nunique()
    .reset_index(name="condition_game_count")
)

In [68]:
# ============================================================
# prelaunch_condition_issue_summary.csv 생성
# ============================================================
# 1행 = 조건값 + 이슈
# 예: genre=Action에서 UI/UX 이슈가 몇 개 게임에서 반복되었는지

condition_issue_base = game_issue_df.merge(
    game_condition_df,
    on=GAME_ID_COL,
    how="left"
)

condition_issue_summary = (
    condition_issue_base
    .groupby(["condition_type", "condition_value", ISSUE_COL])
    .agg(
        issue_game_count=(GAME_ID_COL, "nunique"),
        positive_game_count=("positive_present", "sum"),
        negative_game_count=("negative_present", "sum"),
        mixed_game_count=("mixed_present", "sum"),
        high_urgency_game_count=("high_urgency_present", "sum"),
        both_positive_negative_game_count=("both_positive_negative_present", "sum"),
        total_issue_review_count=("issue_review_count", "sum")
    )
    .reset_index()
)

condition_issue_summary = condition_issue_summary.merge(
    condition_game_count,
    on=["condition_type", "condition_value"],
    how="left"
)

condition_issue_summary["issue_game_ratio"] = (
    condition_issue_summary["issue_game_count"] / condition_issue_summary["condition_game_count"] * 100
).round(1)

condition_issue_summary["priority_level"] = condition_issue_summary.apply(
    lambda row: get_priority_level(
        row["issue_game_count"],
        row["issue_game_ratio"],
        row["negative_game_count"],
        row["high_urgency_game_count"]
    ),
    axis=1
)

condition_issue_summary["priority_reason"] = condition_issue_summary.apply(get_priority_reason, axis=1)

condition_issue_summary = condition_issue_summary.sort_values(
    [
        "condition_type",
        "condition_value",
        "priority_level",
        "issue_game_count",
        "negative_game_count",
        "high_urgency_game_count"
    ],
    ascending=[True, True, True, False, False, False]
)

# 8. LLM 입력용 근거 문장 데이터 생성

In [69]:
# ============================================================
# LLM 근거 문장 생성 함수
# ============================================================
# 나중에 LLM 프롬프트에 바로 넣기 쉬운 문장을 만든다.

def make_evidence_text(row):
    text = (
        f"{row['condition_type']} 조건 '{row['condition_value']}'에서 "
        f"'{row[ISSUE_COL]}' 이슈는 전체 {int(row['condition_game_count'])}개 게임 중 "
        f"{int(row['issue_game_count'])}개 게임에서 반복되었다"
        f"({row['issue_game_ratio']}%). "
        f"부정 맥락은 {int(row['negative_game_count'])}개 게임, "
        f"긍정 맥락은 {int(row['positive_game_count'])}개 게임, "
        f"High urgency는 {int(row['high_urgency_game_count'])}개 게임에서 나타났다. "
        f"우선순위는 '{row['priority_level']}'로 분류된다."
    )
    return text

In [70]:
# ============================================================
# prelaunch_checklist_evidence_base.csv 생성
# ============================================================
# 조건별 이슈 요약 중에서 LLM에게 넘기기 좋은 형태만 정리한다.
# 사용자 조건 필터링은 나중에 LLM 코드에서 수행한다.

checklist_evidence_base = condition_issue_summary.copy()

checklist_evidence_base["llm_evidence_text"] = checklist_evidence_base.apply(make_evidence_text, axis=1)

# 조건별로 너무 많은 이슈가 들어가지 않도록 상위 이슈만 남긴다.
checklist_evidence_base = (
    checklist_evidence_base
    .sort_values(
        ["condition_type", "condition_value", "priority_level", "issue_game_count", "negative_game_count", "high_urgency_game_count"],
        ascending=[True, True, True, False, False, False]
    )
    .groupby(["condition_type", "condition_value"])
    .head(TOP_N_EVIDENCE_PER_CONDITION)
    .reset_index(drop=True)
)

checklist_evidence_base = checklist_evidence_base[
    [
        "condition_type",
        "condition_value",
        ISSUE_COL,
        "condition_game_count",
        "issue_game_count",
        "issue_game_ratio",
        "positive_game_count",
        "negative_game_count",
        "high_urgency_game_count",
        "both_positive_negative_game_count",
        "priority_level",
        "priority_reason",
        "llm_evidence_text"
    ]
]

# 9. CSV 저장

In [71]:
# ============================================================
# CSV 저장
# ============================================================
# 최종 저장 파일은 4개만 만든다.

game_base.to_csv(GAME_BASE_PATH, index=False, encoding="utf-8-sig")
issue_repeat_summary.to_csv(ISSUE_REPEAT_SUMMARY_PATH, index=False, encoding="utf-8-sig")
condition_issue_summary.to_csv(CONDITION_ISSUE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
checklist_evidence_base.to_csv(CHECKLIST_EVIDENCE_BASE_PATH, index=False, encoding="utf-8-sig")

print("CSV 저장 완료")
print(GAME_BASE_PATH)
print(ISSUE_REPEAT_SUMMARY_PATH)
print(CONDITION_ISSUE_SUMMARY_PATH)
print(CHECKLIST_EVIDENCE_BASE_PATH)

CSV 저장 완료
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_allgames_d0-d30\prelaunch_checklist_data\prelaunch_game_base.csv
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_allgames_d0-d30\prelaunch_checklist_data\prelaunch_issue_repeat_summary.csv
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_allgames_d0-d30\prelaunch_checklist_data\prelaunch_condition_issue_summary.csv
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_allgames_d0-d30\prelaunch_checklist_data\prelaunch_checklist_evidence_base.csv


# 10. Tableau 대시보드용 소스 CSV 생성

이번 단계에서는 조원이 Tableau에서 바로 사용할 수 있도록 원천/요약 소스 CSV를 따로 저장한다.

Tableau에서 가장 먼저 쓸 메인 파일은 `tableau_allgames_prelaunch_source.csv`다.


In [72]:
# ============================================================
# 10. Tableau 대시보드용 단일 CSV 생성
# ============================================================

TABLEAU_DIR = OUTPUT_DIR / "tableau_dashboard_csv"
TABLEAU_DIR.mkdir(parents=True, exist_ok=True)

TABLEAU_SOURCE_PATH = TABLEAU_DIR / "tableau_allgames_prelaunch_source.csv"


In [73]:
# ============================================================
# 10-2. 게임-조건-이슈 단위 Tableau 원천 데이터 생성
# ============================================================
# 데이터 단위:
# 1행 = 특정 게임(appid)이 특정 조건값을 가지고 있고,
#       그 게임에서 특정 이슈가 발생한 경우
#
# 예:
# - appid=123, condition_type=genre, condition_value=Action, issue_name_kor=UI/UX
# - appid=123, condition_type=steam_tag, condition_value=Roguelike, issue_name_kor=난이도

tableau_source = game_issue_df.merge(
    game_condition_df,
    on=GAME_ID_COL,
    how="left"
)

# 게임 기본 정보 결합
game_info_cols = [
    GAME_ID_COL,
    GAME_NAME_COL,
    "genres_text",
    "price_group",
    "top_steam_tags_text",
    "categories_text",
    "play_style",
    "review_count",
    "llm_positive_count",
    "llm_negative_count",
    "llm_mixed_count",
    "high_urgency_count",
    "llm_positive_ratio",
    "llm_negative_ratio",
    "high_urgency_ratio",
]

tableau_source = tableau_source.merge(
    game_base[game_info_cols],
    on=[GAME_ID_COL, GAME_NAME_COL],
    how="left"
)


In [74]:
# ============================================================
# 10-3. 조건별 집계값 결합
# ============================================================
# Tableau에서 바로 사용할 수 있도록 조건별 반복 이슈 요약값을 붙인다.
# 단, priority_reason 같은 문장형 설명 컬럼은 제외한다.

condition_summary_for_tableau = condition_issue_summary[
    [
        "condition_type",
        "condition_value",
        ISSUE_COL,
        "condition_game_count",
        "issue_game_count",
        "issue_game_ratio",
        "positive_game_count",
        "negative_game_count",
        "mixed_game_count",
        "high_urgency_game_count",
        "both_positive_negative_game_count",
        "priority_level",
    ]
].copy()

priority_order_map = {
    "상": 1,
    "중": 2,
    "하": 3,
}

condition_summary_for_tableau["priority_order"] = (
    condition_summary_for_tableau["priority_level"].map(priority_order_map)
)

tableau_source = tableau_source.merge(
    condition_summary_for_tableau,
    on=["condition_type", "condition_value", ISSUE_COL],
    how="left"
)

# Tableau에서 보기 쉬운 한글 조건명
condition_type_kor_map = {
    "genre": "장르",
    "price_group": "가격대",
    "steam_tag": "Steam 태그",
    "play_style": "플레이 방식",
}

tableau_source["condition_type_kor"] = tableau_source["condition_type"].map(condition_type_kor_map)


In [75]:
# ============================================================
# 10-4. Steam 태그 DNA 참고 정보 결합
# ============================================================
# 조원 EDA 방식:
# 성과 상위권 게임에서 어떤 Steam 태그가 자주 나타나는지 확인한다.
#
# 이 정보는 "이 태그가 성공을 보장한다"는 의미가 아니라,
# Steam 태그 조건을 해석할 때 쓰는 보조 참고 정보다.
#
# 이 컬럼들은 condition_type == "steam_tag" 행에서만 값이 들어간다.

def parse_tag_keys(tag_text):
    if pd.isna(tag_text):
        return []

    text = str(tag_text).strip()

    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, dict):
            return list(parsed.keys())

        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed]

    except Exception:
        pass

    text = text.replace("[", "").replace("]", "").replace("{", "").replace("}", "")
    text = text.replace("'", "").replace('"', "")
    parts = [part.split(":")[0].strip() for part in text.split(",")]
    return [part for part in parts if part and part.lower() != "nan"]


graded_games = pd.read_csv(GRADED_GAMES_PATH)

graded_games["steam_tag"] = graded_games["tags"].apply(parse_tag_keys)

tag_dna_base = (
    graded_games[
        [
            GAME_ID_COL,
            "name",
            "performance_grade",
            "scale_grade",
            "satisfaction_grade",
            "positive_rate",
            "total_reviews",
            "steam_tag",
        ]
    ]
    .explode("steam_tag")
    .rename(columns={"name": "graded_game_name"})
)

tag_dna_base["steam_tag"] = tag_dna_base["steam_tag"].astype(str).str.strip()
tag_dna_base = tag_dna_base[tag_dna_base["steam_tag"] != ""]
tag_dna_base = tag_dna_base[tag_dna_base["steam_tag"].str.lower() != "nan"]

top_performance_grades = [
    "high_high",
    "high_mid",
    "mid_high",
    "HH",
    "HM",
    "MH",
]

tag_dna_base["is_top_performance_group"] = (
    tag_dna_base["performance_grade"].isin(top_performance_grades).astype(int)
)

tag_dna_summary = (
    tag_dna_base
    .groupby("steam_tag")
    .agg(
        tag_dna_game_count=(GAME_ID_COL, "nunique"),
        top_performance_game_count=("is_top_performance_group", "sum"),
        avg_positive_rate=("positive_rate", "mean"),
        median_total_reviews=("total_reviews", "median"),
    )
    .reset_index()
)

tag_dna_summary["top_performance_ratio"] = (
    tag_dna_summary["top_performance_game_count"]
    / tag_dna_summary["tag_dna_game_count"]
    * 100
).round(1)

tag_dna_summary["avg_positive_rate"] = tag_dna_summary["avg_positive_rate"].round(1)
tag_dna_summary["median_total_reviews"] = tag_dna_summary["median_total_reviews"].round(0)

tableau_source = tableau_source.merge(
    tag_dna_summary,
    left_on="condition_value",
    right_on="steam_tag",
    how="left"
)

# Steam 태그 조건이 아닌 행에서는 태그 DNA 값을 비워둔다.
tag_dna_cols = [
    "tag_dna_game_count",
    "top_performance_game_count",
    "top_performance_ratio",
    "avg_positive_rate",
    "median_total_reviews",
]

for col in tag_dna_cols:
    tableau_source.loc[tableau_source["condition_type"] != "steam_tag", col] = np.nan

tableau_source = tableau_source.drop(columns=["steam_tag"], errors="ignore")


In [76]:
# ============================================================
# 10-5. Tableau용 최종 컬럼 정리 및 저장
# ============================================================
# priority_reason, tableau_note 같은 문장형 설명 컬럼은 제외한다.
# Tableau에서는 숫자/범주형 컬럼 중심으로 필터와 그래프를 구성한다.

tableau_cols = [
    # 게임 식별 정보
    GAME_ID_COL,
    GAME_NAME_COL,

    # 조건 정보
    "condition_type",
    "condition_type_kor",
    "condition_value",

    # 게임 메타 정보
    "genres_text",
    "price_group",
    "top_steam_tags_text",
    "categories_text",
    "play_style",

    # 이슈 정보
    ISSUE_COL,

    # 게임 내부 이슈 발생 정보
    "issue_review_count",
    "positive_review_count",
    "negative_review_count",
    "mixed_review_count",
    "high_urgency_review_count",
    "positive_present",
    "negative_present",
    "mixed_present",
    "high_urgency_present",
    "both_positive_negative_present",

    # 게임 단위 리뷰 요약
    "review_count",
    "llm_positive_count",
    "llm_negative_count",
    "llm_mixed_count",
    "high_urgency_count",
    "llm_positive_ratio",
    "llm_negative_ratio",
    "high_urgency_ratio",

    # 조건별 반복 이슈 집계값
    "condition_game_count",
    "issue_game_count",
    "issue_game_ratio",
    "positive_game_count",
    "negative_game_count",
    "mixed_game_count",
    "high_urgency_game_count",
    "both_positive_negative_game_count",
    "priority_level",
    "priority_order",

    # Steam 태그 DNA 참고 정보
    "tag_dna_game_count",
    "top_performance_game_count",
    "top_performance_ratio",
    "avg_positive_rate",
    "median_total_reviews",
]

tableau_source = tableau_source[tableau_cols].copy()

tableau_source.to_csv(
    TABLEAU_SOURCE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Tableau 대시보드용 단일 CSV 저장 완료")
print(TABLEAU_SOURCE_PATH)
print("데이터 크기:", tableau_source.shape)


Tableau 대시보드용 단일 CSV 저장 완료
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_allgames_d0-d30\tableau_dashboard_csv\tableau_allgames_prelaunch_source.csv
데이터 크기: (17648, 44)


In [77]:
tableau_source.head()

,appid,game_name,condition_type,condition_type_kor,condition_value,genres_text,price_group,top_steam_tags_text,categories_text,play_style,issue_name_kor,issue_review_count,positive_review_count,negative_review_count,mixed_review_count,high_urgency_review_count,positive_present,negative_present,mixed_present,high_urgency_present,both_positive_negative_present,review_count,llm_positive_count,llm_negative_count,llm_mixed_count,high_urgency_count,llm_positive_ratio,llm_negative_ratio,high_urgency_ratio,condition_game_count,issue_game_count,issue_game_ratio,positive_game_count,negative_game_count,mixed_game_count,high_urgency_game_count,both_positive_negative_game_count,priority_level,priority_order,tag_dna_game_count,top_performance_game_count,top_performance_ratio,avg_positive_rate,median_total_reviews
0,571740,Golf It!,price_group,가격대,0-5,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,0,4,0,2,False,True,False,True,False,48,29,15,2,9,60.4,31.2,18.8,54,22,40.7,6,17,0,8,1,상,1,NaN,NaN,NaN,NaN,NaN
1,571740,Golf It!,play_style,플레이 방식,멀티/협동 요소 포함,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,0,4,0,2,False,True,False,True,False,48,29,15,2,9,60.4,31.2,18.8,27,16,59.3,6,14,1,11,4,상,1,NaN,NaN,NaN,NaN,NaN
2,571740,Golf It!,genre,장르,Casual,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,0,4,0,2,False,True,False,True,False,48,29,15,2,9,60.4,31.2,18.8,68,38,55.9,11,34,0,20,7,상,1,NaN,NaN,NaN,NaN,NaN
3,571740,Golf It!,genre,장르,Indie,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,0,4,0,2,False,True,False,True,False,48,29,15,2,9,60.4,31.2,18.8,151,85,56.3,24,77,1,46,17,상,1,NaN,NaN,NaN,NaN,NaN
4,571740,Golf It!,genre,장르,Simulation,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,0,4,0,2,False,True,False,True,False,48,29,15,2,9,60.4,31.2,18.8,42,24,57.1,4,23,0,14,3,상,1,NaN,NaN,NaN,NaN,NaN


tableau_source.head()